In [29]:
import pandas as pd
from util import SEASON_MONTHS, hhmmss_to_hours
import glob
import calendar


historic = pd.read_csv('data/Historic_Flood_Warnings26-raw.csv')

cols = [c for c in historic.columns]
for col in cols:
    unique_vals = historic[col].unique()
    print(f"\n{col} ({len(unique_vals)}): {' | '.join(str(v) for v in unique_vals)}")


DATE (3984): 31/01/2006 | 08/02/2006 | 09/02/2006 | 14/02/2006 | 15/02/2006 | 16/02/2006 | 19/02/2006 | 20/02/2006 | 21/02/2006 | 22/02/2006 | 27/02/2006 | 28/02/2006 | 01/03/2006 | 02/03/2006 | 03/03/2006 | 04/03/2006 | 07/03/2006 | 08/03/2006 | 09/03/2006 | 10/03/2006 | 12/03/2006 | 14/03/2006 | 15/03/2006 | 17/03/2006 | 18/03/2006 | 26/03/2006 | 27/03/2006 | 28/03/2006 | 29/03/2006 | 30/03/2006 | 31/03/2006 | 01/04/2006 | 02/04/2006 | 03/04/2006 | 04/04/2006 | 08/05/2006 | 18/05/2006 | 20/05/2006 | 21/05/2006 | 22/05/2006 | 23/05/2006 | 24/05/2006 | 25/05/2006 | 26/05/2006 | 27/05/2006 | 30/05/2006 | 13/06/2006 | 14/06/2006 | 02/07/2006 | 04/07/2006 | 05/07/2006 | 06/07/2006 | 07/07/2006 | 22/07/2006 | 23/07/2006 | 27/07/2006 | 28/07/2006 | 03/08/2006 | 04/08/2006 | 10/08/2006 | 11/08/2006 | 12/08/2006 | 13/08/2006 | 14/08/2006 | 15/08/2006 | 17/08/2006 | 18/08/2006 | 23/08/2006 | 02/09/2006 | 07/09/2006 | 08/09/2006 | 09/09/2006 | 10/09/2006 | 11/09/2006 | 14/09/2006 | 15/09/2006 

In [30]:
# Remove CODE and WARNING / ALERT AREA NAME columns
historic = historic.drop(columns=['CODE', 'WARNING / ALERT AREA NAME'])

In [31]:
# Filter to only greater manchester
gm_filter = ['Gtr Mancs Mersey and Ches', 'NW - Central', 'NW - North', 'NW - South']
gm_data = historic[historic['AREA'].isin(gm_filter)]
print(gm_data.shape)
print(gm_data['AREA'].value_counts())

(6524, 3)
AREA
NW - North                   3148
Gtr Mancs Mersey and Ches    1407
NW - Central                 1169
NW - South                    800
Name: count, dtype: int64


In [32]:
# Map season to dates
def get_season(month):
    for season, months in SEASON_MONTHS.items():
        if month in months:
            return season

gm_data['DATE'] = pd.to_datetime(gm_data['DATE'], format='%d/%m/%Y')
gm_data['SEASON'] = gm_data['DATE'].dt.month.map(get_season)
gm_data.head()

,DATE,AREA,TYPE,SEASON
87,2006-03-01,NW - North,Flood Watch,Spring
103,2006-03-02,NW - North,Flood Watch,Spring
120,2006-03-03,NW - North,Flood Watch,Spring
142,2006-03-15,NW - South,Flood Watch,Spring
143,2006-03-15,NW - South,Flood Watch,Spring


In [33]:
# Map warning to type
# Assumes update x used to be x
def map_warning(type_val):
    type_lower = str(type_val).lower()
    if 'severe' in type_lower:
        return 'red'
    elif 'alert' in type_lower or 'watch' in type_lower:
        return 'yellow'
    elif 'warning' in type_lower:
        return 'amber'
    return None

gm_data['WARNING'] = gm_data['TYPE'].map(map_warning)
gm_data.head()


,DATE,AREA,TYPE,SEASON,WARNING
87,2006-03-01,NW - North,Flood Watch,Spring,yellow
103,2006-03-02,NW - North,Flood Watch,Spring,yellow
120,2006-03-03,NW - North,Flood Watch,Spring,yellow
142,2006-03-15,NW - South,Flood Watch,Spring,yellow
143,2006-03-15,NW - South,Flood Watch,Spring,yellow


In [34]:
# Filter out duplicates
print(gm_data.duplicated().sum())
gm_data = gm_data.drop_duplicates()

4538


In [35]:
print(gm_data['WARNING'].value_counts())

WARNING
yellow    1509
amber      457
red         20
Name: count, dtype: int64


In [36]:
# Map precipitation to dates
prec = pd.read_csv('data/precipitation.csv')[['time', 'rainfall']]

prec['time'] = pd.to_datetime(prec['time']).dt.date
gm_data['DATE'] = pd.to_datetime(gm_data['DATE'], format='%d/%m/%Y').dt.date

gm_data = gm_data.merge(prec, left_on='DATE', right_on='time', how='left')
gm_data = gm_data.drop(columns=['time']).rename(columns={'rainfall': 'RAINFALL'})
gm_data.head()

,DATE,AREA,TYPE,SEASON,WARNING,RAINFALL
0,2006-03-01,NW - North,Flood Watch,Spring,yellow,0.62715
1,2006-03-02,NW - North,Flood Watch,Spring,yellow,0.09296
2,2006-03-03,NW - North,Flood Watch,Spring,yellow,1.67652
3,2006-03-15,NW - South,Flood Watch,Spring,yellow,0.15601
4,2006-03-26,NW - North,Flood Watch,Spring,yellow,10.13235


In [37]:
# Remove if no precipitation data
gm_data = gm_data.dropna(subset=['RAINFALL'])

In [38]:
# Map soil moisture filepath to date
soil_files = {f.split('dt_smuk_')[1].replace('.tif', ''): f for f in glob.glob('data/soil_moisture/*.tif')}

gm_data['SOIL_MOISTURE_PATH'] = gm_data['DATE'].astype(str).map(soil_files)
gm_data.head()

,DATE,AREA,TYPE,SEASON,WARNING,RAINFALL,SOIL_MOISTURE_PATH
0,2006-03-01,NW - North,Flood Watch,Spring,yellow,0.62715,NaN
1,2006-03-02,NW - North,Flood Watch,Spring,yellow,0.09296,NaN
2,2006-03-03,NW - North,Flood Watch,Spring,yellow,1.67652,NaN
3,2006-03-15,NW - South,Flood Watch,Spring,yellow,0.15601,NaN
4,2006-03-26,NW - North,Flood Watch,Spring,yellow,10.13235,NaN


In [39]:
# Map response time months to dates
response = pd.read_csv('data/response_times.csv')

for col in ['C2_count', 'C3_count']:
    response[col] = response[col].str.replace(',', '').astype(float)

for col in ['C2_mean', 'C3_mean']:
    response[col] = response[col].map(hhmmss_to_hours)

def parse_year_month(year_str, month_str):
    start_year = int(year_str.split('-')[0])
    month_num = list(calendar.month_name).index(month_str)
    year = start_year if month_num >= 4 else start_year + 1
    return (year, month_num)

response['YEAR_MONTH'] = response.apply(lambda r: parse_year_month(r['year'], r['month']), axis=1)

gm_data['YEAR_MONTH'] = gm_data['DATE'].apply(lambda d: (d.year, d.month))

gm_data = gm_data.merge(response[['YEAR_MONTH', 'C2_count', 'C3_count', 'C2_mean', 'C3_mean']], on='YEAR_MONTH', how='left')
gm_data = gm_data.drop(columns=['YEAR_MONTH'])
gm_data.head()

,DATE,AREA,TYPE,SEASON,WARNING,RAINFALL,SOIL_MOISTURE_PATH,C2_count,C3_count,C2_mean,C3_mean
0,2006-03-01,NW - North,Flood Watch,Spring,yellow,0.62715,NaN,NaN,NaN,NaN,NaN
1,2006-03-02,NW - North,Flood Watch,Spring,yellow,0.09296,NaN,NaN,NaN,NaN,NaN
2,2006-03-03,NW - North,Flood Watch,Spring,yellow,1.67652,NaN,NaN,NaN,NaN,NaN
3,2006-03-15,NW - South,Flood Watch,Spring,yellow,0.15601,NaN,NaN,NaN,NaN,NaN
4,2006-03-26,NW - North,Flood Watch,Spring,yellow,10.13235,NaN,NaN,NaN,NaN,NaN


In [40]:
# Map year to dates (for building age, replacing current year)
gm_data['YEAR'] = pd.to_datetime(gm_data['DATE']).dt.year
gm_data.head()

,DATE,AREA,TYPE,SEASON,WARNING,RAINFALL,SOIL_MOISTURE_PATH,C2_count,C3_count,C2_mean,C3_mean,YEAR
0,2006-03-01,NW - North,Flood Watch,Spring,yellow,0.62715,NaN,NaN,NaN,NaN,NaN,2006
1,2006-03-02,NW - North,Flood Watch,Spring,yellow,0.09296,NaN,NaN,NaN,NaN,NaN,2006
2,2006-03-03,NW - North,Flood Watch,Spring,yellow,1.67652,NaN,NaN,NaN,NaN,NaN,2006
3,2006-03-15,NW - South,Flood Watch,Spring,yellow,0.15601,NaN,NaN,NaN,NaN,NaN,2006
4,2006-03-26,NW - North,Flood Watch,Spring,yellow,10.13235,NaN,NaN,NaN,NaN,NaN,2006


In [41]:
# Output final csvs to sample from
for warning in gm_data['WARNING'].unique():
    subset = gm_data[gm_data['WARNING'] == warning]
    subset.to_csv(f'warning/{warning}_warnings.csv', index=False)